

Download the labeled training dataset from this link: House Prices - Advanced Regression Techniques

Part 1: PCA and Variance Threshold in a Linear Regression

Part 2: Categorical Feature Selection

Download the data from this link Mushroom Classification. Based on several categorical features, you will predict whether or not a mushroom is edible or poisonous. Import the data as a data frame and ensure it is loaded correctly. Convert the categorical features (all of them) to dummy variables. Split the data into a training and test set. Fit a decision tree classifier on the training set. Report the accuracy and create a confusion matrix for the model prediction on the test set. Create a visualization of the decision tree. Use a χ2-statistic selector to pick the five best features for this data (see section 10.4 of the Machine Learning with Python Cookbook). Which five features were selected in step 7? Hint: Use the get_support function. Repeat steps 4 and 5 with the five best features selected in step 7. Summarize your findings.

In [ ]:
#1 Import the housing data as a data frame and ensure that the data is loaded properly.

import pandas as pd

# read in the housing data as a dataframe
# I have the csv in the same directory for ease
housing = pd.read_csv("train.csv",)

# ensure the data is loaded properly
housing.head()

In [ ]:
# 2 Drop the "Id" column and any features that are missing more than 40% of their values.

# Returns a list of column names with > 40% missing values
missing_cols = housing.columns[housing.isnull().mean() > 0.40].tolist()

print(f"These columns are missing more than 40% of their values: \n {missing_cols}")

In [ ]:
# 3 Drop the "Id" column and the columns identified above as missing
housing.drop(columns=['Id', 'Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature'], inplace=True)
housing.head()

In [ ]:
# For numerical columns, fill in any missing data with the median value.
housing = housing.fillna(housing.median(numeric_only=True))

# 4 For categorical columns, fill in any missing data with the most common value (mode).
# make the list of categorical columns
cat_cols = housing.select_dtypes(include=['object']).columns
housing[cat_cols] = housing[cat_cols].fillna(housing[cat_cols].mode().iloc[0])
housing.head()

In [ ]:
# 5 Convert the categorical columns to dummy variables.
housing_dummies = pd.get_dummies(housing, columns=cat_cols, drop_first=True)
housing_dummies.head()

In [ ]:
# 6 Split the data into a training and test set, where the SalePrice column is the target.
from sklearn.model_selection import train_test_split

# isolate the target column (drop this)
X = housing_dummies.drop('SalePrice', axis=1) # features
y = housing['SalePrice'] # target

# split into training/test 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=73)


# 7 Run a linear regression and report the R2-value and RMSE on the test set.
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

# create model
linear_model = LinearRegression()
# train
linear_model.fit(X_train, y_train)

# predict based on test data
y_pred = linear_model.predict(X_test)

# calculate metrics
linear_r2 = r2_score(y_test, y_pred)

# get RMSE
linear_rmse = root_mean_squared_error(y_test, y_pred)

print("*** Baseline Linear Model ***")
print(f"R^2 Score: {linear_r2:.4f}")
print(f"RMSE: {linear_rmse:.4f}")

**Interpretation**

The  of  means that approximately  of the variability in house prices is able to be explained by this model. This suggests a significant level of overfitting, or what is deemed as noise, due to multicollinear features. This has had an impact on the predictive power of the coefficients and

The RMSE score measures the average magnitude of the error, so for the target variable of dollars, this means that, on average, the model's predictions deviate from the actual sale prices by about $66,299.

In [ ]:
# 8 Fit and transform the training features with a PCA so that 90% of the variance is retained
# (see section 9.1 in the Machine Learning with Python Cookbook).
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

# standardize the features (Fit on TRAIN, transform both)
# Note: PCA is highly sensitive to scaling.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit PCA on scaled training data only
pca = PCA(n_components=0.90, whiten=True)
X_train_pca = pca.fit_transform(X_train_scaled)

# 10 Transform but DO NOT fit the test features with the same PCA.
X_test_pca = pca.transform(X_test_scaled)

# 9 Report the feature reduction
print("*** Feature Reduction Report ***")
print(f"Original number of features: {X_train_scaled.shape[1]}")
print(f"Reduced number of features after PCA: {X_train_pca.shape[1]}")

In [ ]:
# 11 Repeat step 7 with your PCA transformed data.
# Run a linear regression and report the R2-value and RMSE on the test set.

# run Linear Regression on PCA components
y_pca_pred = pca_model.predict(X_test_pca)

# metrics for PCA model
pca_r2 = r2_score(y_test, y_pca_pred)
pca_rmse = root_mean_squared_error(y_test, y_pca_pred)

print("*** PCA Transformed Data Results ***")
print(f"R^2 Score: {pca_r2:.4f}")
print(f"RMSE: {pca_rmse:.4f}")

**Interpretation**

This $R^2$ score is much better for this model using the PCA transformed data. This means that approximately $86.7%$ of the variability in house prices is able to be explained by this model. 

The RMSE score measures the average magnitude of the error, so for the target variable of dollars, this means that, on average, the model's predictions deviate from the actual sale prices by about $28,359.

In [ ]:
# 12 Take your original training features (from step 6) and apply a min-max scaler to them.
# Scale features between 0 and 1
min_max = MinMaxScaler()
X_train_mm = min_max.fit_transform(X_train)
X_test_mm = min_max.transform(X_test)

# 13 Find the min-max scaled features in your training set that have a variance above 0.1
# Drop features with variance below 0.1
selector = VarianceThreshold(threshold=0.1)

# 14 Transform but DO NOT fit the test features with the same steps applied in steps 11 and 12.
X_train_high_var = selector.fit_transform(X_train_mm)
X_test_high_var = selector.transform(X_test_mm)


# 15 Repeat step 7 with the high variance data.
# Regression on high variance features
y_var_pred = var_model.predict(X_test_high_var)
var_r2 = r2_score(y_test, y_var_pred)
var_rmse = root_mean_squared_error(y_test, y_var_pred)

print("*** Variance Threshold Model ***")
print(f"R^2 Score: {var_r2:.4f}")
print(f"RMSE: {var_rmse:.4f}")

**Interpretation**

This $R^2$ score is better for this model than the baseline model, but not as good as the model that utilized the PCA transformed data. This means that approximately $68.9%$ of the variability in house prices is able to be explained by this model. 

The RMSE score measures the average magnitude of the error, so for the target variable of dollars, this means that, on average, the model's predictions deviate from the actual sale prices by about $43,342.


### Summarize your findings

| Model | Number of Features | $R^2$ Score | RMSE |
| -- | -- | -- | -- |
| LinearRegression | 238 | 0.2721 | 66299.3587 |
| PCA (90% Variance) | 126 | 0.8668 | 28358.7513 |
| Variance Threshold (>0.1) | 40 | 0.6889 | 43342.8793 |


The PCA-transformed model significantly outperformed the baseline Linear Regression and Threshold models. This demonstrates that for high-dimensional data like housing prices, capturing the linear combinations of features that explain the most variance (dimensionality reduction) is more effective than simply discarding features with low individual variance. The high $R^2$ of $0.8668$ indicates that the principal components are highly representative of the price-determining factors. 


## Part 2: Categorical Feature Selection

Download the data from this link [Mushroom Classification](https://www.kaggle.com/datasets/uciml/mushroom-classification). Based on several categorical features, you will predict whether or not a mushroom is edible or poisonous.

1. Import the data as a data frame and ensure it is loaded correctly.
2. Convert the categorical features (all of them) to dummy variables.
3. Split the data into a training and test set.
4. Fit a decision tree classifier on the training set.
5. Report the accuracy and create a confusion matrix for the model prediction on the test set.
6. Create a visualization of the decision tree.
7. Use a χ2-statistic selector to pick the five best features for this data (see section 10.4 of the Machine Learning with Python Cookbook).
8. Which five features were selected in step 7? Hint: Use the get_support function.
9. Repeat steps 4 and 5 with the five best features selected in step 7.
10. Summarize your findings.